# 05 — Preprocessing Pipeline


In [ ]:
import sys
from pathlib import Path
REPO = Path.cwd()
for candidate in [REPO, REPO.parent, REPO.parent.parent]:
    if (candidate / "src" / "neuro").exists():
        REPO = candidate
        break
sys.path.insert(0, str(REPO / "src"))
import os
os.chdir(REPO)
%matplotlib inline


In [ ]:

import numpy as np
import pandas as pd
from pathlib import Path
from neuro.config import PROCESSED_DIR
from neuro.bids import validate_bids
from neuro.features import get_schaefer_masker, extract_roi_timeseries
from nilearn.image import smooth_img
import nibabel as nib
import matplotlib.pyplot as plt
from neuro import viz

report = validate_bids()
row = report["runs"][report["runs"]["bold_exists"]].iloc[0]
raw = nib.load(row["bold_path"])
smoothed = smooth_img(raw, fwhm=6)
masker = get_schaefer_masker(n_rois=50)
ts_clean = masker.fit_transform(smoothed)
np.save(PROCESSED_DIR / "example_preprocessed_ts.npy", ts_clean)
print("Preprocessed shape:", ts_clean.shape)


In [ ]:

# Before/after smoothing — middle volume
mid = raw.shape[3] // 2
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(raw.get_fdata()[:, :, raw.shape[2]//2, mid], cmap="gray")
ax[0].set_title("Raw BOLD (mid slice/vol)")
ax[1].imshow(smoothed.get_fdata()[:, :, smoothed.shape[2]//2, mid], cmap="gray")
ax[1].set_title("Smoothed 6mm FWHM")
viz.savefig("05_preprocessing_before_after.png")


In [ ]:

from pathlib import Path
nb_name = Path.cwd().name if False else "05_preprocessing_pipeline"
!jupyter nbconvert --to html notebooks/05_preprocessing_pipeline.ipynb --output-dir notebooks/html 2>/dev/null || true
